# Flang Bounds Sanitizer - Rapid Colab Build
This notebook downloads LLVM, integrates your bounds checking pass, and compiles it using Colab's cloud infrastructure.
Because Colab sessions stay alive, if you re-run the build cell, it will only compile the files that changed (taking seconds instead of hours)!

In [ ]:
!apt-get update
!apt-get install -y cmake ninja-build build-essential

In [ ]:
!rm -rf llvm-project project
!git clone --depth 1 https://github.com/llvm/llvm-project.git
!git clone https://github.com/vishwapanchal/Flang-Bounds-Sanitizer.git project

In [ ]:
%%bash
cp project/src/pass/BoundsCheckInstrumentation.* llvm-project/flang/lib/Optimizer/Transforms/
sed -i '/add_flang_library(FIRTransforms/a \  BoundsCheckInstrumentation.cpp' llvm-project/flang/lib/Optimizer/Transforms/CMakeLists.txt

RUNTIME_DIR=$(find llvm-project -type d -path "*/flang-rt/lib/runtime" -o -path "*/flang/runtime" | head -n 1)
if [ -n "$RUNTIME_DIR" ]; then
  cp project/src/runtime/bounds-check.* "$RUNTIME_DIR/"
  if [[ "$RUNTIME_DIR" == *"flang-rt"* ]]; then
    sed -i '/add_flangrt_library(flang_rt.runtime/a \  bounds-check.cpp' "$RUNTIME_DIR/CMakeLists.txt"
  else
    sed -i '/add_flang_library(FlangRuntime/a \  bounds-check.cpp' "$RUNTIME_DIR/CMakeLists.txt"
  fi
fi

sed -i '/pm.addPass(hlfir::createLowerHLFIRIntrinsics());/i \  pm.addPass(fir::createHLFIRBoundsCheckPass());' llvm-project/flang/lib/Optimizer/Passes/Pipelines.cpp
echo "Integration Complete!"

In [ ]:
%%bash
cd llvm-project
mkdir -p build && cd build
# Aggressive optimizations to build as fast as possible
cmake -G Ninja ../llvm \
  -DCMAKE_BUILD_TYPE=Release \
  -DLLVM_ENABLE_PROJECTS="flang;mlir" \
  -DLLVM_ENABLE_RUNTIMES="flang-rt" \
  -DLLVM_TARGETS_TO_BUILD="X86" \
  -DLLVM_BUILD_TESTS=OFF \
  -DLLVM_BUILD_EXAMPLES=OFF \
  -DLLVM_INCLUDE_TESTS=OFF \
  -DLLVM_INCLUDE_EXAMPLES=OFF

In [ ]:
%%bash
cd llvm-project/build
ninja flang flang-rt || ninja flang FlangRuntime || ninja flang

In [ ]:
%%bash
cd llvm-project/build
bin/flang -fc1 -emit-hlfir ../../project/src/tests/bounds_check.f90 -o bounds_check.hlfir
cat bounds_check.hlfir | grep "_FortranABoundsCheck"